# Lesson 4: Persistence and Streaming

In [1]:
from dotenv import load_dotenv

_ = load_dotenv()

In [2]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_tavily import TavilySearch

In [3]:
tool = TavilySearch(max_results=2)

In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.add_edge(START, "llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [7]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
model = ChatOpenAI(model="gpt-4o")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [8]:
messages = [HumanMessage(content="What is the weather in sf?")]

In [9]:
thread = {"configurable": {"thread_id": "1"}}

In [10]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1259, 'total_tokens': 1283, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSf3Q2PINirPpKyrubocrJJbNhKxC', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d715c-3f9f-7270-9df3-52673e585452-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'San Francisco current weather', 'search_depth': 'fast'}, 'id': 'call_THnuSICQatQ6ZVe4fXfGtQ2x', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1259, 'output_tokens': 24, 'total_tokens': 1283, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_d

In [11]:
messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 1919, 'total_tokens': 1943, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 1792}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSf3Zf3xchARsvqRwmjxAjNd39NBG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d715c-65de-7921-88c9-3070a02c8a08-0', tool_calls=[{'name': 'tavily_search', 'args': {'query': 'Los Angeles current weather', 'search_depth': 'fast'}, 'id': 'call_dkjPqUPlCKdgONmy0ZZgTsRq', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 1919, 'output_tokens': 24, 'total_tokens': 1943, 'input_token_details': {'audio': 0, 'cache_read': 1792}

In [12]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Based on the current weather information, San Francisco has a temperature of around 23 degrees Celsius (about 73 degrees Fahrenheit), while there is no specific temperature mentioned for Los Angeles. However, considering Los Angeles's reputation for warmer weather and the description of being ideal for outdoor activities, it may be warmer in general. Without specific temperature data for Los Angeles, it is not possible to definitively conclude which city is warmer at this moment.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 2492, 'total_tokens': 2580, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 2432}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSf3haUGNlI

In [13]:
messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content="Could you please clarify what you're comparing to determine which is warmer? For example, are you asking about regions, clothing materials, seasons, or something else?", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 1257, 'total_tokens': 1289, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSf3j3KphjcsBppKNdRsCwgajyAbP', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d715c-913a-7a72-99c8-4c388e6aadb2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1257, 'output_tokens': 32, 'total_tokens': 1289, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'o

## Streaming tokens

In [14]:
memory = MemorySaver()
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [15]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

Calling: {'name': 'tavily_search', 'args': {'query': 'San Francisco weather', 'include_images': False}, 'id': 'call_ICkmz2zo3eZk4QgS3r7wxnST', 'type': 'tool_call'}
Back to the model!
The| current| weather| in| San| Francisco| is| over|cast| with| a| temperature| of| |12|.|8|°C| (|55|.|0|°F|).| The| wind| is| coming| from| the| west|-s|outh|west| at| |3|.|8| mph| (|6|.|1| k|ph|),| and| the| humidity| is| at| |96|%.| The| visibility| is| |16|.|0| km| (|9|.|0| miles|),| and| there| is| no| precipitation|.|